# Step 07.1 — `nn.Embedding`：把 token ID 真正接入 MLX 参数系统

Step 6.2 已经手工理解：

```text
token ID -> embedding table 的某一行 -> C 维向量
```

现在正式使用 MLX：

```python
nn.Embedding(V, C)
```

这一节完整解决 5 个问题：

1. `nn.Embedding(V,C)` 创建了什么对象和参数；
2. 为什么 `embedding(ids)` 本质上是 row lookup / gather；
3. 为什么不用真的构造 One-Hot，以及这样能省多少内存；
4. `(T,)`、`(B,T)` 经过 Embedding 后 shape 怎样增加一个 `C` 维；
5. 训练时 gradient 到底更新谁，以及它和 MiniGPT 的 token embedding `wte` 是什么关系。

## 1. `nn.Embedding(V,C)` 到底创建了什么？

假设 vocabulary 有：

```text
V = 6
```

我们希望每个 token 用：

```text
C = 4
```

个连续特征表示。

于是：

```python
token_embedding = nn.Embedding(6, 4)
```

会创建一个 `nn.Embedding` Module，其中最核心的 parameter 是：

```text
token_embedding.weight.shape = (6,4) = (V,C)
```

也就是 6 行，每一行对应一个 token；每行有 4 个可学习数值。

In [ ]:
import mlx.core as mx
import mlx.nn as nn

V = 6
C = 4

token_embedding = nn.Embedding(V, C)

print("module type :", type(token_embedding))
print("weight shape:", token_embedding.weight.shape)
print("parameters  :", token_embedding.parameters())
print("parameter count:", token_embedding.weight.size)

参数量非常直接：

```text
V * C = 6 * 4 = 24
```

和 `nn.Linear` 对比一下：

```text
Linear.weight    -> 一个线性变换矩阵
Embedding.weight -> 一张“token ID -> vector”的可学习查表
```

`Embedding` 没有必要额外准备一个 bias；每一行向量本身就已经是该 token 的完整可学习表示。

初始化时这些数值还没有语言语义，只是初始参数；真正的结构要通过训练形成。

## 2. `embedding(ids)` 本质上就是从 `weight` 里按 ID 取行

假设输入：

```text
ids = [2, 0, 2, 5]
```

那么逻辑上做的是：

```text
id 2 -> weight[2]
id 0 -> weight[0]
id 2 -> weight[2]
id 5 -> weight[5]
```

所以输入 token 数量不变，但每个整数 ID 被替换成一个长度为 `C` 的向量。

In [ ]:
ids = mx.array([2, 0, 2, 5])

via_module = token_embedding(ids)
via_direct_lookup = token_embedding.weight[ids]

mx.eval(via_module, via_direct_lookup)

print("ids shape           :", ids.shape)
print("embedding output    :", via_module.shape)
print("direct lookup output:", via_direct_lookup.shape)
print("same:", bool(mx.allclose(via_module, via_direct_lookup)))

应该得到：

```text
same: True
```

这说明 `nn.Embedding` 在 forward 语义上没有神秘操作：它就是根据整数 indices 从 `weight` 取对应 rows。

### 重复 token 为什么得到完全相同的向量？

上面的 `ids` 里 token 2 出现两次：

```text
[2, 0, 2, 5]
 ↑     ↑
```

它们都会读取同一个：

```text
weight[2]
```

所以此刻两处 token 2 的 **token embedding** 完全一样。

In [ ]:
first_token2 = via_module[0]
second_token2 = via_module[2]

mx.eval(first_token2, second_token2)

print("first token-2 vector :", first_token2)
print("second token-2 vector:", second_token2)
print("identical:", bool(mx.allclose(first_token2, second_token2)))

这里先埋一个以后非常重要的概念：

> **同一个 token 的初始 token embedding 相同，但经过 position information、Attention 和后续网络后，同一个 token 在不同上下文中的 hidden state 可以完全不同。**

所以“Embedding 向量”和“最终上下文化表示”不是一回事。

## 3. 为什么真实实现不构造 One-Hot？——数学等价，但工程成本差很多

Step 6.2 已经证明单个 token：

```text
one_hot(id) @ E == E[id]
```

如果整个 batch：

```text
ids.shape = (B,T)
```

真的转成 One-Hot，就会变成：

```text
(B,T,V)
```

再与：

```text
E.shape = (V,C)
```

相乘得到 `(B,T,C)`。

数学没问题，但当 `V` 很大时非常浪费。

In [ ]:
# 一个更接近真实语言模型量级的内存直觉
B = 32
T = 128
V_large = 50_000
C_large = 768

one_hot_values = B * T * V_large
id_values = B * T
embedding_output_values = B * T * C_large

fp32_bytes = 4
int32_bytes = 4

one_hot_mb = one_hot_values * fp32_bytes / 1024**2
ids_kb = id_values * int32_bytes / 1024
embedding_output_mb = embedding_output_values * fp32_bytes / 1024**2

print(f"explicit one-hot : {one_hot_mb:.1f} MB")
print(f"integer IDs      : {ids_kb:.1f} KB")
print(f"embedding output : {embedding_output_mb:.1f} MB")

这个例子中，显式 One-Hot 本身就大约需要数百 MB，而 `(B,T)` integer IDs 只有几十 KB 量级。

因此框架直接做 **gather / lookup**：

```text
integer IDs
   ↓
从 E 取需要的 rows
   ↓
(B,T,C)
```

这不仅概念更直接，也特别符合我们这个 Apple Silicon 项目强调的低内存目标。

### 用 heatmap 看 `Embedding.weight` 是一张什么表

每一行是 token，每一列是 embedding feature dimension。

`imshow()` 很适合把这种二维 parameter matrix 画成 heatmap；`colorbar()` 用来说明数值大小对应的颜色尺度。

In [ ]:
import matplotlib.pyplot as plt

mx.eval(token_embedding.weight)
weight_for_plot = token_embedding.weight.tolist()

fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(weight_for_plot, aspect="auto")
ax.set_xlabel("embedding dimension C")
ax.set_ylabel("token ID / row")
ax.set_title("Embedding weight table: one row per token")
fig.colorbar(image, ax=ax, label="parameter value")
plt.show()

## 4. Shape 规则：Embedding 是在输入 shape 后面追加一个 `C`

这是 `nn.Embedding` 最值得形成肌肉记忆的 shape 规则。

如果输入一个 scalar token ID：

```text
() -> (C,)
```

如果输入一条 sequence：

```text
(T,) -> (T,C)
```

如果输入一个 batch：

```text
(B,T) -> (B,T,C)
```

更一般地：

> **Embedding 保留所有原来的 index axes，并在最后追加 embedding dimension `C`。**

In [ ]:
single_id = mx.array(3)
sequence_ids = mx.array([1, 4, 2, 1])
batch_ids = mx.array([
    [1, 4, 2, 1],
    [5, 0, 3, 2],
])

single_vec = token_embedding(single_id)
sequence_vecs = token_embedding(sequence_ids)
batch_vecs = token_embedding(batch_ids)

print("scalar ID  :", single_id.shape, "->", single_vec.shape)
print("sequence   :", sequence_ids.shape, "->", sequence_vecs.shape)
print("batch      :", batch_ids.shape, "->", batch_vecs.shape)

对 MiniGPT 最重要的是：

```text
token IDs
(B,T)
   ↓ nn.Embedding(V,C)
token vectors
(B,T,C)
```

这一步不会改变 `B` 和 `T`；它只是把每个原本的离散整数位置替换成一个 `C` 维连续向量。

所以 `(B,T,C)` 里的三个轴现在有非常具体的来源：

```text
B -> batch sampler 产生
T -> block / sequence positions
C -> embedding lookup 新增的 feature axis
```

## 5. Training 时谁被更新？——ID 不求梯度，Embedding rows 才是 Parameter

输入：

```text
ids = [2,0,2,5]
```

这些整数是**数据 / indices**，不是连续 trainable parameters。

训练时真正被 optimizer 修改的是：

```text
token_embedding.weight
```

尤其是 forward 中实际被访问到的 rows 会参与当前 loss 的计算图。

概念上：

```text
ID 2 ──┐
       ├──> weight[2] ──> downstream model ──> loss
ID 2 ──┘
```

如果同一个 token 在 batch 中出现多次，同一个 `weight[2]` 会被重复使用；反向传播时这些使用路径对该 row 的 gradient contribution 会汇总到同一组 parameter 上。

这就是 parameter sharing 在 Embedding 中的具体形式。

### 接到我们的 MiniGPT 命名

经典 GPT 风格代码里常把 token embedding 命名为：

```python
self.wte = nn.Embedding(vocab_size, n_embd)
```

可以把 `wte` 理解成 **word/token embedding** 的历史命名。

我们的字符级模型虽然 token 是 character，数学结构完全一样：

```text
wte.weight.shape = (vocab_size, n_embd)
```

如果：

```text
vocab_size = 65
n_embd = 128
```

则：

```text
wte.weight.shape = (65,128)
parameter count  = 65*128 = 8,320
```

### 7.1 最终结论

- `nn.Embedding(V,C)` 是一个 `nn.Module`，核心 parameter 是 `weight.shape=(V,C)`；
- `embedding(ids)` 在 forward 语义上就是按 integer IDs gather `weight` 的 rows；
- 相同 token ID 在 token-embedding 阶段读取同一行，因此得到同一初始 token vector；
- 显式 One-Hot 与 lookup 数学等价，但实际 lookup 避免构造巨大 `(B,T,V)` sparse-like tensor；
- Embedding 保留输入 index shape，并在末尾追加 `C`：`(B,T)->(B,T,C)`；
- token IDs 是数据，不是可训练连续 parameter；训练修改的是 embedding weight rows；
- 同一 token 多次出现时，共享同一 row，gradient contributions 汇总到同一 parameter；
- 下一节 7.2：真正研究 embedding geometry——距离、dot product、cosine similarity 到底分别衡量什么，以及哪些“语义聚类”说法可以讲、哪些不能乱讲。